# O6 · Modelo estrella de vehículos eléctricos

## Objetivo
Construir un modelo estrella en Gold para analizar los vehículos por fabricante y tipo.

## Tablas
- `dim_fabricante`: una fila por fabricante.
- `dim_tipo_vehiculo`: una fila por tipo de vehículo.
- `fact_vehiculos`: una fila por vehículo identificado por `vehicle_id`.

## Fuente
Los datos provienen de las tablas depuradas de Silver.

## Grano
La tabla de hechos representa un vehículo dentro del extracto utilizado.
No representa ventas ni un histórico de cambios del vehículo.

## Claves
Las claves primarias y foráneas documentan las relaciones del modelo.
En Databricks estas restricciones son informativas, por lo que se comprobará
la integridad mediante consultas de validación.

In [0]:
%sql
-- Crear y cargar la dimensión de fabricantes.
-- Generar un identificador estable a partir del nombre depurado en Silver.
-- Actualizar el contenido sin acumular filas entre ejecuciones.

CREATE TABLE IF NOT EXISTS pf_fallas.gold.dim_fabricante (
    fabricante_id STRING NOT NULL,
    make STRING NOT NULL,
    CONSTRAINT pk_dim_fabricante PRIMARY KEY (fabricante_id)
)
USING DELTA
COMMENT 'Dimensión de fabricantes. Una fila por fabricante presente en Silver.';

INSERT OVERWRITE TABLE pf_fallas.gold.dim_fabricante
SELECT DISTINCT
    SHA2(make, 256) AS fabricante_id,
    make
FROM pf_fallas.silver.vehiculos
WHERE make IS NOT NULL;

In [0]:
%sql
-- Crear y cargar la dimensión de tipos de vehículo.
-- Utilizar el código BEV o PHEV como clave primaria.
-- Actualizar el contenido desde el catálogo de tipos existente en Silver.

CREATE TABLE IF NOT EXISTS pf_fallas.gold.dim_tipo_vehiculo (
    type_code STRING NOT NULL,
    type_description STRING NOT NULL,
    CONSTRAINT pk_dim_tipo_vehiculo PRIMARY KEY (type_code)
)
USING DELTA
COMMENT 'Dimensión de tipos de vehículo eléctrico y sus descripciones.';

INSERT OVERWRITE TABLE pf_fallas.gold.dim_tipo_vehiculo
SELECT DISTINCT
    type_code,
    type_description
FROM pf_fallas.silver.tipos_vehiculo;

In [0]:
%sql
-- Verificar la cantidad de filas, la unicidad de las claves
-- y la ausencia de claves nulas en ambas dimensiones.

SELECT
    'dim_fabricante' AS tabla,
    COUNT(*) AS filas_totales,
    COUNT(DISTINCT fabricante_id) AS claves_unicas,
    COUNT(*) - COUNT(DISTINCT fabricante_id) AS claves_repetidas_o_nulas
FROM pf_fallas.gold.dim_fabricante

UNION ALL

SELECT
    'dim_tipo_vehiculo' AS tabla,
    COUNT(*) AS filas_totales,
    COUNT(DISTINCT type_code) AS claves_unicas,
    COUNT(*) - COUNT(DISTINCT type_code) AS claves_repetidas_o_nulas
FROM pf_fallas.gold.dim_tipo_vehiculo;

In [0]:
-- Crear la tabla de hechos con una fila por vehículo.
-- Declarar la clave primaria y las relaciones con las dos dimensiones.

CREATE TABLE IF NOT EXISTS pf_fallas.gold.fact_vehiculos (
    vehicle_id BIGINT NOT NULL,
    fabricante_id STRING NOT NULL,
    type_code STRING NOT NULL,
    model_year INT,
    electric_range_valid INT,
    CONSTRAINT pk_fact_vehiculos PRIMARY KEY (vehicle_id),
    CONSTRAINT fk_vehiculos_fabricante
        FOREIGN KEY (fabricante_id)
        REFERENCES pf_fallas.gold.dim_fabricante (fabricante_id),
    CONSTRAINT fk_vehiculos_tipo
        FOREIGN KEY (type_code)
        REFERENCES pf_fallas.gold.dim_tipo_vehiculo (type_code)
)
USING DELTA
COMMENT 'Una fila por vehículo del extracto, identificado por vehicle_id.';

In [0]:
-- Cargar los vehículos depurados de Silver y obtener sus claves dimensionales.
-- Reemplazar el contenido para evitar acumulación entre ejecuciones.
-- Conservar todos los vehículos de origen mediante LEFT JOIN.

INSERT OVERWRITE TABLE pf_fallas.gold.fact_vehiculos
SELECT
    v.vehicle_id,
    f.fabricante_id,
    t.type_code,
    v.model_year,
    v.electric_range_valid
FROM pf_fallas.silver.vehiculos AS v
LEFT JOIN pf_fallas.gold.dim_fabricante AS f
    ON v.make = f.make
LEFT JOIN pf_fallas.gold.dim_tipo_vehiculo AS t
    ON v.type_code = t.type_code;

In [0]:
-- Verificar que exista una fila por vehículo y que todas las claves
-- tengan una correspondencia en sus respectivas dimensiones.

SELECT
    COUNT(*) AS filas_totales,
    COUNT(DISTINCT v.vehicle_id) AS vehiculos_unicos,
    COUNT(*) - COUNT(DISTINCT v.vehicle_id) AS registros_repetidos,
    COUNT_IF(f.fabricante_id IS NULL) AS fabricantes_sin_correspondencia,
    COUNT_IF(t.type_code IS NULL) AS tipos_sin_correspondencia
FROM pf_fallas.gold.fact_vehiculos AS v
LEFT JOIN pf_fallas.gold.dim_fabricante AS f
    ON v.fabricante_id = f.fabricante_id
LEFT JOIN pf_fallas.gold.dim_tipo_vehiculo AS t
    ON v.type_code = t.type_code;

In [0]:
-- Analizar la distribución de vehículos utilizando la tabla de hechos
-- y ambas dimensiones del modelo estrella.

SELECT
    f.make,
    t.type_code,
    t.type_description,
    COUNT(*) AS total_vehiculos,
    CAST(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER ()
        AS DECIMAL(7,2)
    ) AS porcentaje_total
FROM pf_fallas.gold.fact_vehiculos AS v
INNER JOIN pf_fallas.gold.dim_fabricante AS f
    ON v.fabricante_id = f.fabricante_id
INNER JOIN pf_fallas.gold.dim_tipo_vehiculo AS t
    ON v.type_code = t.type_code
GROUP BY
    f.make,
    t.type_code,
    t.type_description
ORDER BY total_vehiculos DESC, f.make, t.type_code;

In [0]:
-- Comprobar que el modelo estrella reproduce el agregado Gold existente.
-- El resultado esperado es cero diferencias.

WITH modelo_estrella AS (
    SELECT
        f.make,
        t.type_code,
        COUNT(*) AS total_vehiculos,
        CAST(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER ()
            AS DECIMAL(7,2)
        ) AS porcentaje_total
    FROM pf_fallas.gold.fact_vehiculos AS v
    INNER JOIN pf_fallas.gold.dim_fabricante AS f
        ON v.fabricante_id = f.fabricante_id
    INNER JOIN pf_fallas.gold.dim_tipo_vehiculo AS t
        ON v.type_code = t.type_code
    GROUP BY f.make, t.type_code
),
solo_en_modelo AS (
    SELECT make, type_code, total_vehiculos, porcentaje_total
    FROM modelo_estrella
    EXCEPT ALL
    SELECT make, type_code, total_vehiculos, porcentaje_total
    FROM pf_fallas.gold.vehiculos_por_fabricante
),
solo_en_gold_actual AS (
    SELECT make, type_code, total_vehiculos, porcentaje_total
    FROM pf_fallas.gold.vehiculos_por_fabricante
    EXCEPT ALL
    SELECT make, type_code, total_vehiculos, porcentaje_total
    FROM modelo_estrella
)
SELECT
    (SELECT COUNT(*) FROM solo_en_modelo) AS diferencias_en_modelo,
    (SELECT COUNT(*) FROM solo_en_gold_actual) AS diferencias_en_gold_actual;

In [0]:
-- Mostrar las tres claves primarias y las dos claves foráneas
-- que documentan la estructura del modelo estrella.

SELECT
    table_name,
    constraint_name,
    constraint_type
FROM pf_fallas.information_schema.table_constraints
WHERE table_schema = 'gold'
  AND table_name IN (
      'dim_fabricante',
      'dim_tipo_vehiculo',
      'fact_vehiculos'
  )
  AND constraint_type IN ('PRIMARY KEY', 'FOREIGN KEY')
ORDER BY table_name, constraint_type, constraint_name;